# Explore richer structures · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 22. Explore richer structures

Use multiple index levels, list-valued columns, and categorical encodings.

Before you start: Understand indexes, reshaping, and basic data types.

Official reference: https://pandas.pydata.org/docs/user_guide/advanced.html

### M22C01. Create a MultiIndex

A MultiIndex stores more than one label per row. Here Campus and Student_ID jointly locate a student. sort_index arranges the levels for predictable selection.

**Read:** What does the tuple ("RAK", "S003") identify?

Answer: A row with Campus RAK and Student_ID S003.

**Predict:** What Name and Score will appear?

Answer: Chloe and 91.

**Explain:** Why keep Student_ID as an additional level?

Answer: Campus alone is shared by several students; the second level identifies the individual record.



In [ ]:
# Predict before running: What Name and Score will appear?
df = students().set_index(["Campus", "Student_ID"]).sort_index()
result = df.loc[("RAK", "S003"), ["Name", "Score"]]
print(result)

**Your turn:** Use xs to select all Dubai students at the Campus level and print Name and Score.

For the check, keep these result names: `result`.

**Hint:** Call xs("Dubai", level="Campus") on the indexed table.

**Watch for this:** A tuple used as a MultiIndex key is different from a list of separate row keys.

In [ ]:
# Change this code to complete the task.
# Question: Use xs to select all Dubai students at the Campus level and print Name and Score.
df = students().set_index(["Campus", "Student_ID"]).sort_index()
result = df.xs("Dubai", level="Campus")[["Name", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 6
assert result.index.name == "Student_ID"
assert "S002" in result.index
print("Code checks passed. Review the written explanation as well.")

The selection contains six Dubai students; xs drops the selected level by default.

**Quick check:** What is a MultiIndex?

1. Several labels per row or column axis entry
2. Several unrelated DataFrames
3. A duplicate-free column

Answer: 1. It represents an axis with multiple levels of labels.

**Explain what you learned:** Name index levels so selections are understandable.

### M22C02. Stack and unstack

stack moves column labels into an additional row-index level. unstack moves an index level back into columns. The explicit future_stack=True selects the modern stack implementation on pandas 2.2 and 3.x.

**Read:** How many index levels does the stacked result have?

Answer: Two: Student_ID and the original measure column label.

**Predict:** How many values are in the stacked result?

Answer: 4, from two students and two measures.

**Explain:** Were the measurements averaged during stacking?

Answer: No. Their labels moved; the original values remain.



In [ ]:
# Predict before running: How many values are in the stacked result?
wide = students().head(2).set_index("Student_ID")[["Score", "Attendance"]]
result = wide.stack(future_stack=True)
print(result)

**Your turn:** Stack the first three students and unstack the last level back into a DataFrame. Check that it equals the original wide table.

For the check, keep these result names: `result`, `wide`.

**Hint:** Use .unstack() on the stacked Series, then .equals(wide).

**Watch for this:** Missing combinations can appear when reshaping sparse data; inspect the resulting shape.

In [ ]:
# Change this code to complete the task.
# Question: Stack the first three students and unstack the last level back into a DataFrame. Check that it equals the original wide table.
wide = students().head(3).set_index("Student_ID")[["Score", "Attendance"]]
stacked = wide.stack(future_stack=True)
result = stacked.unstack().reindex(columns=wide.columns)
print(result)
print("Values restored:", result.equals(wide))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.equals(wide)
assert result.shape == (3, 2)
print("Code checks passed. Review the written explanation as well.")

The explicit column reindex restores the original column order for the equality check.

**Quick check:** Which operation returns the last index level to columns?

1. unstack
2. drop_duplicates
3. head

Answer: 1. unstack reshapes one index level into column labels.

**Explain what you learned:** Stack and unstack move labels between axes.

### M22C03. Expand lists with explode

Sometimes one cell contains several items, such as a list of skills. explode creates a separate row for each item and repeats the other columns. This changes the unit of analysis from one row per student to one row per student-skill.

**Read:** What kind of value is stored in each Skills cell before explode?

Answer: A Python list.

**Predict:** How many rows result?

Answer: 3: two skills for S001 and one skill for S002.

**Explain:** Why would counting rows after explode overcount students?

Answer: A student with multiple skills appears in multiple rows.



In [ ]:
# Predict before running: How many rows result?
df = pd.DataFrame({"Student_ID": ["S001", "S002"], "Skills": [["Python", "SQL"], ["Excel"]]})
result = df.explode("Skills", ignore_index=True)
print(result)

**Your turn:** Give S002 a second skill, SQL. Explode the table, then store the count of student-skill rows for each skill in result as a value_counts() Series.

For the check, keep these result names: `result`.

**Hint:** After explode, use the Skills column value_counts().

**Watch for this:** An empty list becomes a row with a missing item; a string that looks like a list is still a string.

In [ ]:
# Change this code to complete the task.
# Question: Give S002 a second skill, SQL. Explode the table, then store the count of student-skill rows for each skill in result as a value_counts() Series.
df = pd.DataFrame({"Student_ID": ["S001", "S002"], "Skills": [["Python", "SQL"], ["Excel", "SQL"]]})
long = df.explode("Skills", ignore_index=True)
result = long["Skills"].value_counts()
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["SQL"] == 2
assert result.sum() == 4
print("Code checks passed. Review the written explanation as well.")

SQL appears twice; Python and Excel appear once each.

**Quick check:** After explode, one row represents what?

1. Always one unique student
2. One listed item belonging to a student
3. One average skill score

Answer: 2. Each item becomes a row; the student identifier can repeat.

**Explain what you learned:** Check what a row represents after changing the layout.

### M22C04. Order and encode categories

Categorical values can follow a chosen order instead of alphabetical order. get_dummies makes one indicator column per category. These indicator values describe membership, not a numerical ranking.

**Read:** Which Program sorts first?

Answer: IT, because it is listed first in the ordered categories.

**Predict:** What are Program_IT and Program_AI for S001?

Answer: 0 and 1 respectively.

**Explain:** Does a dummy value of 1 mean AI is better than IT?

Answer: No. It only marks whether that row belongs to the AI category.



In [ ]:
# Predict before running: What are Program_IT and Program_AI for S001?
df = students().head(4).copy()
df["Program"] = pd.Categorical(df["Program"], categories=["IT", "AI"], ordered=True)
ordered = df.sort_values("Program", kind="stable")
result = pd.get_dummies(ordered["Program"], prefix="Program", dtype=int)
print(ordered[["Student_ID", "Program"]])
print(result)

**Your turn:** Change the category order to AI then IT and print the sorted student IDs and dummy columns.

For the check, keep these result names: `ordered`, `result`.

**Hint:** Reverse the categories list before sorting.

**Watch for this:** Values outside a specified category list become missing; validate allowed categories first.

In [ ]:
# Change this code to complete the task.
# Question: Change the category order to AI then IT and print the sorted student IDs and dummy columns.
df = students().head(4).copy()
df["Program"] = pd.Categorical(df["Program"], categories=["AI", "IT"], ordered=True)
ordered = df.sort_values("Program", kind="stable")
result = pd.get_dummies(ordered["Program"], prefix="Program", dtype=int)
print(ordered[["Student_ID", "Program"]])
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert ordered["Student_ID"].tolist() == ["S001", "S003", "S002", "S004"]
assert result.loc[0, "Program_AI"] == 1
print("Code checks passed. Review the written explanation as well.")

AI students now appear first; each row still has one indicator value equal to 1.

**Quick check:** What does get_dummies encode?

1. Category membership
2. Student achievement
3. A guaranteed useful order

Answer: 1. Each column indicates membership in one category.

**Explain what you learned:** Category order and indicator encoding are separate choices.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates